In [ ]:
!pip -q install -U transformers peft accelerate
%cd /content
!rm -rf /content/DanyAI
!git clone -q https://github.com/DanyaVoredom/DanyAI.git /content/DanyAI
%cd /content/DanyAI
!pip -q install -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv || true
import torch
print("torch", torch.__version__, torch.cuda.is_available())
print("setup done")

In [ ]:
import os, pathlib, urllib.request
try:
    os.chdir("/content/DanyAI")
except Exception:
    pass
base = "https://github.com/DanyaVoredom/DanyAI/releases/download/data-v2/"
for name in ["train.bin", "val.bin", "tokenizer.json", "meta.json", "sft.jsonl", "tools.jsonl"]:
    dst = pathlib.Path("data", name)
    dst.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(base + name, dst)
    print("got", name, dst.stat().st_size)
print("data", sorted(os.listdir("data")))

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u tools/finetune_lora.py --model huihui-ai/Huihui-Qwen3.5-2B-abliterated --data data/sft.jsonl,data/tools.jsonl --out /content/danyai_runs/lora --max-len 512 --batch 2 --grad-accum 8 --lr 1e-4 --warmup 30 --epochs 1 --eval-share 0.005 --eval-every 100 --limit 0 --fallback Qwen/Qwen2.5-1.5B-Instruct
print("lora done")

In [ ]:
import base64, pathlib, time, uuid, urllib.request, zipfile
src = pathlib.Path("/content/danyai_runs/lora")
zp = pathlib.Path("/content/danyai_runs/lora.zip")
if not src.exists():
    print("@@MISSING /content/danyai_runs/lora")
else:
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(src.rglob("*")):
            if f.is_file():
                z.write(f, f.relative_to(src))
    data = zp.read_bytes()
    print("zip bytes", len(data), flush=True)

    def post(url, fields, field, filename, payload, timeout=600):
        boundary = "----danyai" + uuid.uuid4().hex
        body = b""
        for k, v in fields:
            body += ("--" + boundary + "\r\nContent-Disposition: form-data; name=\"" + k + "\"\r\n\r\n" + v + "\r\n").encode()
        body += ("--" + boundary + "\r\nContent-Disposition: form-data; name=\"" + field + "\"; filename=\"" + filename + "\"\r\nContent-Type: application/octet-stream\r\n\r\n").encode()
        body += payload + ("\r\n--" + boundary + "--\r\n").encode()
        req = urllib.request.Request(url, data=body, headers={"Content-Type": "multipart/form-data; boundary=" + boundary})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.read().decode(errors="replace").strip()

    urls = []
    for attempt in range(3):
        try:
            u = post("https://litterbox.catbox.moe/resources/internals/api.php",
                     [("reqtype", "fileupload"), ("time", "72h")], "fileToUpload", "lora.zip", data)
            if u.startswith("http"):
                urls.append(("litterbox", u))
                break
        except Exception as e:
            print("litterbox fail", type(e).__name__, str(e)[:120], flush=True)
            time.sleep(5 * (attempt + 1))
    if not urls:
        try:
            u = post("https://0x0.st", [], "file", "lora.zip", data)
            if u.startswith("http"):
                urls.append(("0x0", u.split()[0]))
        except Exception as e:
            print("0x0 fail", type(e).__name__, str(e)[:120], flush=True)
    if not urls:
        try:
            u = post("https://catbox.moe/user/api.php",
                     [("reqtype", "fileupload")], "fileToUpload", "lora.zip", data)
            if u.startswith("http"):
                urls.append(("catbox", u))
        except Exception as e:
            print("catbox fail", type(e).__name__, str(e)[:120], flush=True)
    for host, u in urls:
        print("URL", host, u, flush=True)
    if not urls:
        b = base64.b64encode(data).decode()
        print("@@BEGIN /content/danyai_runs/lora.zip " + str(len(b)))
        for i in range(0, len(b), 50000):
            print(b[i:i + 50000])
        print("@@END /content/danyai_runs/lora.zip")
print("pack lora done")